In [ ]:
import sys
import glob
import torch
import json
import glob
import sys
import os

import seaborn as sns

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from scipy.stats import norm
from collections import defaultdict

sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats

from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

sys.path.append('../utils/')
sys.path.append('../src/model/')

sys.path.append("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/")

from utils.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir
from utils.dprime import recompute_dprime_by_isi_per_subject
from utils.reliability import compute_itemwise_split_half_reliability

from utils.loading import load_results, load_results_with_isi0_exclusion, load_results_with_isi0_dprime_exclusion, move_sequences_to_used, load_results_with_exclusion
from utils.dprime import recompute_dprime_by_isi_per_subject
from utils.reliability import compute_itemwise_split_half_reliability
from utils.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir

from utils.reliability import compute_power_curve
from utils.plotting import plot_power_curve

import DistanceMemoryModel
import encoders

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fit_sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/*wav")
test_sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025/*wav")

In [ ]:
pc_dims = 256

pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=pc_dims,
    model_params=model_params,
    sr=20000,
    rms_level=0.05,
    duration=2.0,
    device=device
)

zscore_projector = encoders.ZScoreSpace(pc_texture_model, device=device)
zscore_projector.fit(fit_sounds_list)

In [ ]:
# -*- coding: utf-8 -*-
"""
Run-averaged clean-probe vs drifting-bank analysis on a grid of (noise_variance × criterion).

For each (nv, crit) we run K stochastic drifts and aggregate:
  - avg #neighbors < criterion vs time (mean ± SEM across runs)
  - P(yes | T) where "yes" := min_j ||clean[i]-bank_T[j]||_2 < criterion (mean ± SEM across runs)
  - distribution of min distances vs time (mean probability across runs)

Produces THREE grid figures (rows=noise_variances, cols=criteria):
  1) Average neighbors vs time (mean ± SEM)
  2) P(yes | T) (mean ± SEM)
  3) Distribution of min distances vs time (mean probability)
"""

from typing import Dict, Tuple, List
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

# ----------------------------- CONFIG -----------------------------
T_max: int = 120                                 # inclusive time steps per run
noise_variances: List[float] = [0.16, 0.165, 0.17] # 4*0.6 / np.sqrt(256), 8*0.6 / np.sqrt(256),  12*0.6 / np.sqrt(256),
criteria: List[float] = [10, 11, 12]
n_runs: int = 4                                 # independent drift runs per (nv, crit)
base_seed: int = 1234                             # reproducible seeds: base_seed + r

# Distance histogram bins (for the distribution grids)
dist_min: float = 0.0
dist_max: float = 100.0
n_bins: int = 60
# ------------------------------------------------------------------

def stack_bank(bank: List[torch.Tensor]) -> torch.Tensor:
    """Stack a list of 1D reps into [N, D] on their existing device/dtype."""
    return torch.stack([rep.view(-1) for rep in bank], dim=0)

@torch.no_grad()
def run_clean_vs_drifting_one(
    memory_model,
    T_max: int,
    criterion: float,
    clean_bank_snap: List[torch.Tensor],
) -> Tuple[torch.Tensor, torch.Tensor]:
    """One stochastic drift run; returns (min_dists_TN, counts_below_TN), both [T_max+1, N]."""
    clean_bank = [rep.detach().clone() for rep in clean_bank_snap]   # frozen probes + restore target
    device, dtype = clean_bank[0].device, clean_bank[0].dtype
    N = len(clean_bank)
    X0 = stack_bank(clean_bank)                                      # [N, D] frozen probes

    # fresh working copy so each run starts clean
    memory_model.memory_bank = [rep.detach().clone() for rep in clean_bank]

    min_dists_TN    = torch.empty(T_max + 1, N, device=device, dtype=dtype)
    counts_below_TN = torch.empty(T_max + 1, N, device=device, dtype=torch.int32)

    for T in range(T_max + 1):
        XT = stack_bank(memory_model.memory_bank)                    # [N, D] current bank
        D  = torch.cdist(X0, XT, p=2)                                # [N, N] distances
        min_dists_TN[T]    = D.min(dim=1).values                     # includes self j=i
        counts_below_TN[T] = (D < criterion).sum(dim=1)              # includes self if < crit
        if T < T_max:
            memory_model.apply_noise_to_memory()

    memory_model.memory_bank = clean_bank                             # restore
    return min_dists_TN, counts_below_TN

def time_distance_hist(min_dists_TN: torch.Tensor, bin_edges: np.ndarray) -> np.ndarray:
    """Row-normalized histogram over distances per time step -> [T_max+1, n_bins]."""
    md = min_dists_TN.detach().cpu().numpy()                         # [T, N]
    T_rows = md.shape[0]
    H = np.zeros((T_rows, len(bin_edges) - 1), dtype=np.float32)
    for t in range(T_rows):
        hist, _ = np.histogram(md[t], bins=bin_edges)
        tot = hist.sum()
        H[t] = hist / tot if tot > 0 else hist
    return H

def main():
    # Precompute bins for the 2D histograms
    bin_edges = np.linspace(dist_min, dist_max, n_bins + 1)

    # results[(nv, crit)] stores aggregated arrays
    results: Dict[Tuple[float, float], Dict[str, np.ndarray]] = {}

    # Grid loops
    for nv in noise_variances:
        # Construct a fresh model per noise variance and load data
          # <- your list of sounds


        for crit in criteria:

            memory_model = DistanceMemoryModel.DistanceMemoryModel(
                encoding_model=zscore_projector,
                noise_variance=float(nv),
                criterion=crit,                 # internal criterion unused here
                device='cuda'
            )
            memory_model._fill_memory_bank(test_sounds_list)    

                        # Snapshot clean probes once for this nv
            with torch.no_grad():
                clean_bank_snap = [rep.detach().clone() for rep in memory_model.memory_bank]

            # Per-run accumulators
            run_avg_neighbors: List[np.ndarray] = []                  # [R, T]
            run_yes_prob: List[np.ndarray]   = []                     # [R, T]
            run_Hs: List[np.ndarray]         = []                     # [R, T, n_bins]

            for r in range(n_runs):
                # Reproducible noise across runs
                torch.manual_seed(base_seed + r)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(base_seed + r)

                # One drift run
                min_dists_TN, counts_below_TN = run_clean_vs_drifting_one(
                    memory_model, T_max, float(crit), clean_bank_snap
                )

                # Avg neighbors across probes at each T
                avg_neighbors_T = counts_below_TN.float().mean(dim=1).cpu().numpy()   # [T]
                run_avg_neighbors.append(avg_neighbors_T)

                # P(yes | T): fraction of probes with min distance < crit at each T
                yes_prob_T = (min_dists_TN < float(crit)).float().mean(dim=1).cpu().numpy()  # [T]
                run_yes_prob.append(yes_prob_T)

                # Distribution of min distances vs time
                H = time_distance_hist(min_dists_TN, bin_edges)                          # [T, n_bins]
                run_Hs.append(H)

            # Stack across runs and compute mean ± SEM
            avg_neighbors_stack = np.stack(run_avg_neighbors, axis=0)                    # [R, T]
            yes_prob_stack      = np.stack(run_yes_prob, axis=0)                         # [R, T]
            H_stack             = np.stack(run_Hs, axis=0)                               # [R, T, n_bins]

            results[(nv, crit)] = {
                "avg_neighbors_mean": avg_neighbors_stack.mean(axis=0),                  # [T]
                "avg_neighbors_sem":  avg_neighbors_stack.std(axis=0, ddof=1) / np.sqrt(n_runs),
                "yes_mean":           yes_prob_stack.mean(axis=0),                       # [T]
                "yes_sem":            yes_prob_stack.std(axis=0, ddof=1) / np.sqrt(n_runs),
                "H_mean":             H_stack.mean(axis=0),                              # [T, n_bins]
            }

    rows, cols = len(noise_variances), len(criteria)
    T_axis = np.arange(T_max + 1)

    # ---------------------------
    # Figure 1: avg neighbors grid (mean ± SEM)
    # ---------------------------
    fig1, axes1 = plt.subplots(rows, cols,
                               figsize=(4*cols, 3*rows),
                               sharex=True, sharey=True)
    if rows == 1 and cols == 1:
        axes1 = np.array([[axes1]])
    elif rows == 1 or cols == 1:
        axes1 = axes1.reshape(rows, cols)

    for i, nv in enumerate(noise_variances):
        for j, crit in enumerate(criteria):
            ax = axes1[i, j]
            out = results[(nv, crit)]
            mean = out["avg_neighbors_mean"]
            sem  = out["avg_neighbors_sem"]
            ax.plot(T_axis, mean)
            ax.fill_between(T_axis, mean - sem, mean + sem, alpha=0.25)
            ax.set_title(f"nv={nv:g}, c={crit:g}")
            if i == rows - 1:
                ax.set_xlabel("T")
            if j == 0:
                ax.set_ylabel("avg neighbors")
    fig1.suptitle(f"Average neighbors < criterion vs time (mean ± SEM across {n_runs} runs)", y=0.995)
    fig1.tight_layout()

    # ---------------------------
    # Figure 2: P(yes | T) grid (mean ± SEM)
    # ---------------------------
    fig2, axes2 = plt.subplots(rows, cols,
                               figsize=(4*cols, 3*rows),
                               sharex=True, sharey=True)
    if rows == 1 and cols == 1:
        axes2 = np.array([[axes2]])
    elif rows == 1 or cols == 1:
        axes2 = axes2.reshape(rows, cols)

    for i, nv in enumerate(noise_variances):
        for j, crit in enumerate(criteria):
            ax = axes2[i, j]
            out = results[(nv, crit)]
            mean = out["yes_mean"]
            sem  = out["yes_sem"]
            ax.plot(T_axis, mean)
            ax.fill_between(T_axis, mean - sem, mean + sem, alpha=0.25)
            ax.set_ylim(-0.02, 1.02)
            ax.set_title(f"nv={nv:g}, c={crit:g}")
            if i == rows - 1:
                ax.set_xlabel("T")
            if j == 0:
                ax.set_ylabel("P(yes)")
    fig2.suptitle(f"P(yes | T) (mean ± SEM across {n_runs} runs)", y=0.995)
    fig2.tight_layout()

    plt.show()

if __name__ == "__main__":
    main()
    

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt

T_max = 120
nv = 0.165 
D = 10# your noise_variance
ISI = 16
crit = nv*np.sqrt(D*ISI)


memory_model = DistanceMemoryModel.DistanceMemoryModel(
    encoding_model=zscore_projector,
    noise_variance=float(nv),
    criterion=crit,                 # internal criterion unused here
    device='cuda'
)
memory_model._fill_memory_bank(test_sounds_list)  

# snapshot clean bank
clean_bank = [rep.detach().clone() for rep in memory_model.memory_bank]
X0 = torch.stack([rep.view(-1) for rep in clean_bank], dim=0)  # [N,D]
D = X0.shape[1]

N = len(clean_bank)
dists = torch.zeros(T_max+1, N, device=X0.device)

for T in range(T_max+1):
    XT = torch.stack([rep.view(-1) for rep in memory_model.memory_bank], dim=0)
    diff = (XT - X0).norm(dim=1)        # [N], L2 drift from origin
    dists[T] = diff
    if T < T_max:
        memory_model.apply_noise_to_memory()

dists_np = dists.cpu().numpy()
mean = dists_np.mean(axis=1)
sem  = dists_np.std(axis=1) / np.sqrt(N)

# theoretical curve: expected L2 ≈ σ * sqrt(T * D)
theory = nv * np.sqrt(np.arange(T_max+1) * D)

# plot
plt.figure(figsize=(7,5))
plt.plot(range(T_max+1), mean, label="empirical mean", color="C0")
plt.fill_between(range(T_max+1), mean-sem, mean+sem, color="C0", alpha=0.3)
plt.plot(range(T_max+1), theory, "--", label="theory σ√(T·D)", color="C1")
plt.xlabel("T")
plt.ylabel("||x_T - x_0||")
plt.axvline(x=16, alpha=0.3)
plt.axhline(y=nv*np.sqrt(D*ISI), alpha=0.3, label=f"σ√(16·D) = {nv*np.sqrt(D*ISI)}, criterion={crit} ")

plt.title(f"Brownian drift of individual items [Euclidean Distance] (nl={nv}, D={D})")
plt.legend()
plt.show()

In [ ]:
# -*- coding: utf-8 -*-
"""
Diagnose bimodality by separating self vs nearest-other distances over time.

What it does:
- Freezes CLEAN probes X0 from the model's memory bank at T=0.
- Repeatedly applies memory_model.apply_noise_to_memory() to drift the bank.
- At each T, computes for every probe i:
    d_self[i]  = || X0[i] - XT[i] ||_2
    d_other[i] = min_{j!=i} || X0[i] - XT[j] ||_2
    d_min[i]   = min(d_self[i], d_other[i])
- Plots:
  A) Histograms of d_self vs d_other at selected T with vertical criterion.
  B) Time courses: mean±SEM of d_self and d_other; frac(other<self); P(d_min<criterion).

Assumptions:
- You have DistanceMemoryModel, zscore_projector, and test_sounds_list available.
- memory_model.noise_variance is the per-dimension std (naming quirk in your class).
"""

import math
import numpy as np
import torch
import matplotlib.pyplot as plt

# --------------------- CONFIG (edit) ---------------------
nv     = 0.165         # noise std per dim used by your model
crit   = np.sqrt(D*ISI*4) * nv         # decision criterion for YES
T_max  = 120          # simulate up to this many drift steps
times_to_plot = [0, 5, 10, 16, 50, 120]   # histogram snapshots
scatter_T = 16        # which T to show in the scatter panel
device = 'cuda'       # 'cuda' or 'cpu'
# ---------------------------------------------------------

# 1) Build model and fill memory bank (clean state)
memory_model = DistanceMemoryModel.DistanceMemoryModel(
    encoding_model=zscore_projector,
    noise_variance=float(nv),
    criterion=float(crit),
    device=device
)
memory_model._fill_memory_bank(test_sounds_list)

# 2) Freeze clean probes X0 and reset working bank to clean
with torch.no_grad():
    clean_bank = [rep.detach().clone() for rep in memory_model.memory_bank]
    X0 = torch.stack([rep.view(-1) for rep in clean_bank], dim=0)  # [N,D]
    memory_model.memory_bank = [rep.detach().clone() for rep in clean_bank]

N, D = X0.shape

# 3) Storage tensors over time
d_self_TN  = torch.empty(T_max+1, N, device=X0.device, dtype=X0.dtype)    # self distances
d_other_TN = torch.empty(T_max+1, N, device=X0.device, dtype=X0.dtype)    # nearest-other
d_min_TN   = torch.empty(T_max+1, N, device=X0.device, dtype=X0.dtype)    # overall min
other_beats_self_T = torch.empty(T_max+1, device=X0.device, dtype=X0.dtype)  # fraction other<self
p_yes_T    = torch.empty(T_max+1, device=X0.device, dtype=X0.dtype)       # P(min<crit)

eye_bool = None  # reuse mask across T once N is known

# 4) Main loop: compute distances at each T, then drift once
with torch.no_grad():
    for T in range(T_max+1):
        XT = torch.stack([rep.view(-1) for rep in memory_model.memory_bank], dim=0)  # [N,D]
        Dmat = torch.cdist(X0, XT, p=2)                                              # [N,N]

        if eye_bool is None:
            eye_bool = torch.eye(N, device=Dmat.device, dtype=torch.bool)

        # self distances = diagonal
        d_self = torch.diag(Dmat)                              # [N]

        # nearest-other = min over columns with diagonal masked to +inf
        D_other = Dmat.clone()
        D_other[eye_bool] = float('inf')
        d_other = D_other.min(dim=1).values                    # [N]

        # overall min and summary stats
        d_min = torch.minimum(d_self, d_other)                 # [N]
        d_self_TN[T]  = d_self
        d_other_TN[T] = d_other
        d_min_TN[T]   = d_min

        other_beats_self_T[T] = (d_other < d_self).float().mean()          # identity confusion rate
        p_yes_T[T]            = (d_min < crit).float().mean()              # decision probability

        if T < T_max:
            memory_model.apply_noise_to_memory()

# 5) FIGURE A: Histograms of self vs other at selected T with criterion line
cols = len(times_to_plot)
figA, axes = plt.subplots(1, cols, figsize=(4.2*cols, 3.6), sharey=True)
if cols == 1:
    axes = [axes]

# choose common x-limits by high quantile of pooled distances among selected Ts
pooled = np.concatenate([torch.cat([d_self_TN[t], d_other_TN[t]]).cpu().numpy()
                         for t in times_to_plot])
x_max = float(np.percentile(pooled, 99.5))
for ax, T in zip(axes, times_to_plot):
    s = d_self_TN[T].cpu().numpy()
    o = d_other_TN[T].cpu().numpy()
    ax.hist(s, bins=40, range=(0, x_max), density=True, alpha=0.55, label="self")
    ax.hist(o, bins=40, range=(0, x_max), density=True, alpha=0.55, label="nearest-other")
    ax.axvline(crit, color="red", linestyle="--", lw=2, label=f"criterion={crit:g}")
    ax.set_title(f"T={T} | P(other<self)={other_beats_self_T[T].item():.2f}\nP(min<crit)={p_yes_T[T].item():.2f}")
    ax.set_xlabel("distance")
    if T == times_to_plot[0]:
        ax.set_ylabel("density")
    ax.legend(fontsize=8, loc="upper right")
figA.suptitle("Self vs Nearest-Other Distance Distributions (selected T)", y=0.98)
plt.tight_layout()
plt.show()



In [ ]:
# 6) FIGURE B: Time courses – means ± SEM, identity confusion, and decision probability
taxis = np.arange(T_max+1)
s_mean = d_self_TN.mean(dim=1).cpu().numpy()
s_sem  = (d_self_TN.std(dim=1, unbiased=True)/math.sqrt(N)).cpu().numpy()
o_mean = d_other_TN.mean(dim=1).cpu().numpy()
o_sem  = (d_other_TN.std(dim=1, unbiased=True)/math.sqrt(N)).cpu().numpy()
confuse = other_beats_self_T.cpu().numpy()
pyes    = p_yes_T.cpu().numpy()

figB, axs = plt.subplots(3, 1, figsize=(8, 9), sharex=True)

# Means ± SEM
axs[0].plot(taxis, s_mean, label="self")
axs[0].fill_between(taxis, s_mean-s_sem, s_mean+s_sem, alpha=0.25)
axs[0].plot(taxis, o_mean, label="nearest-other")
axs[0].fill_between(taxis, o_mean-o_sem, o_mean+o_sem, alpha=0.25)
axs[0].axhline(crit, color="red", linestyle="--", lw=1, label="criterion")
axs[0].set_ylabel("distance")
axs[0].set_title("Mean ± SEM of Self vs Nearest-Other")
axs[0].legend()

# Identity confusion rate
axs[1].plot(taxis, confuse)
axs[1].set_ylabel("P(other < self)")
axs[1].set_ylim(-0.02, 1.02)
axs[1].set_title("Identity Confusion Rate (nearest-other beats self)")

# Decision probability
axs[2].plot(taxis, pyes)
axs[2].set_ylabel("P(min < criterion)")
axs[2].set_xlabel("T (time steps)")
axs[2].set_ylim(-0.02, 1.02)
axs[2].set_title("Decision Probability vs Time")

plt.tight_layout()
plt.show()



In [ ]:
# 7) (Optional) FIGURE C: Scatter of self vs other at a chosen T
T = int(scatter_T)
s = d_self_TN[T].cpu().numpy()
o = d_other_TN[T].cpu().numpy()
plt.figure(figsize=(5.5,5))
plt.scatter(s, o, alpha=0.6, s=16)
plt.axvline(crit, color="red", linestyle="--", lw=1)
plt.axhline(crit, color="red", linestyle="--", lw=1)
plt.plot([0, max(s.max(), o.max())], [0, max(s.max(), o.max())], 'k--', lw=1)  # y=x line
plt.xlabel("self distance")
plt.ylabel("nearest-other distance")
plt.title(f"Self vs Nearest-Other at T={T}\npoints below red lines → YES; below y=x → other beats self")
plt.tight_layout()
plt.show()

In [ ]:
# -*- coding: utf-8 -*-
"""
Progressive insertion with drift (no functions; linear script).
- Encodes clean reps X0 once (no drift).
- Repeats: random insertion order; each step drifts current bank once, then
  measures distances from incoming CLEAN probe to CURRENT (drifted) bank, logs min distance,
  then inserts the CLEAN probe into the bank.
- Aggregates per-step distributions across permutations and plots time courses + histograms.

Assumptions:
- DistanceMemoryModel.DistanceMemoryModel, zscore_projector, and test_sounds_list exist.
- memory_model.noise_variance is the per-dimension std (naming quirk).
"""

# ----------------------------- Imports (top) -----------------------------
import numpy as np
import torch
import matplotlib.pyplot as plt
import math
import random

# ----------------------------- Config -----------------------------------
nv = 0.165                            # noise std per dim used by your model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
N_PERMS = 1000                          # how many random insertion orders
SEED = 1234                           # reproducibility; set to None to randomize
times_to_plot = [1, 5, 10, 20, 40, 80]  # 1-indexed insertion steps for histograms

# Criterion in distance units (simple normative choice).
# Scale of one-step noise in L2 ~ nv*sqrt(D); here we pick a multiple (tune as needed).
CRIT_MULT = 1.5

# -------------------------- RNG seeding ---------------------------------
if SEED is not None:
    np.random.seed(SEED)
    random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

# ------------------- Encode CLEAN representations once ------------------
# Build a temp model, fill its bank to leverage your existing pipeline.
_tmp_model = DistanceMemoryModel.DistanceMemoryModel(
    encoding_model=zscore_projector,
    noise_variance=float(nv),
    criterion=0.0,            # unused here
    device=device
)
_tmp_model._fill_memory_bank(test_sounds_list)

with torch.no_grad():  # freeze clean bank
    clean_bank = [rep.detach().clone() for rep in _tmp_model.memory_bank]
    X0 = torch.stack([rep.view(-1) for rep in clean_bank], dim=0).to(device)  # [N,D]
del _tmp_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

N, D = X0.shape
crit = np.sqrt(D*ISI) * nv  # distance threshold; tune via calibration if needed

# -------------------- Allocate storage for aggregates -------------------
# mins_by_t[t, p] = min distance at step t (1..N) on permutation p (NaN if bank empty)
mins_by_t = np.full((N, N_PERMS), np.nan, dtype=np.float64)
# p_yes_by_t[t] = fraction of permutations where min < crit at step t
p_yes_by_t = np.zeros(N, dtype=np.float64)

# ------------------------ Main simulation loop -------------------------
for p in range(N_PERMS):
    # Fresh model with matching noise/criterion; start with EMPTY bank
    memory_model = DistanceMemoryModel.DistanceMemoryModel(
        encoding_model=zscore_projector,
        noise_variance=float(nv),
        criterion=float(crit),
        device=device
    )
    memory_model.memory_bank = []  # ensure empty

    # Random insertion order of indices 0..N-1
    order = np.arange(N)
    np.random.shuffle(order)

    with torch.no_grad():
        for t, idx in enumerate(order, start=1):  # step t = 1..N
            # Drift the CURRENT bank by ONE step (Brownian-like)
            if len(memory_model.memory_bank) > 0:
                memory_model.apply_noise_to_memory()

            # Incoming probe is CLEAN representation (no noise yet)
            probe = X0[idx]  # [D]

            # Compute distances from probe to CURRENT (drifted) bank
            if len(memory_model.memory_bank) == 0:
                min_dist = np.nan                        # empty bank on first insertion
                said_yes = False
            else:
                # Stack bank to [n,D] on device
                bank_mat = torch.stack([rep.view(-1) for rep in memory_model.memory_bank], dim=0)
                # L2 distances probe->each bank item
                dists = torch.linalg.norm(bank_mat - probe.view(1, -1), dim=1)
                min_dist = float(dists.min().item())
                said_yes = bool(min_dist < crit)

            # Log per-step outputs
            mins_by_t[t - 1, p] = min_dist
            if said_yes:
                p_yes_by_t[t - 1] += 1.0

            # Insert the CLEAN probe into bank (store a clone so later drift doesn't alias)
            memory_model.memory_bank.append(probe.detach().clone())

# Convert counts to probabilities
p_yes_by_t /= N_PERMS

# Compute aggregate mean±SEM of min distances (ignore NaNs from step 1)
mean_min_by_t = np.nanmean(mins_by_t, axis=1)
sem_min_by_t = np.array([
    (np.nanstd(mins_by_t[t], ddof=1) / np.sqrt(np.sum(~np.isnan(mins_by_t[t]))))
    if np.sum(~np.isnan(mins_by_t[t])) > 1 else np.nan
    for t in range(N)
], dtype=np.float64)

# ------------------------------ Plots -----------------------------------
# 1) Time course: mean±SEM(min distance) and P(min<crit)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# Left: mean±SEM of min distances vs insertion step
x = np.arange(1, N + 1)
axes[0].plot(x, mean_min_by_t, label='mean min distance')
axes[0].fill_between(
    x,
    mean_min_by_t - sem_min_by_t,
    mean_min_by_t + sem_min_by_t,
    alpha=0.25,
    label='± SEM'
)
axes[0].axhline(crit, linestyle='--', linewidth=2, label=f'criterion = {crit:.3g}')
axes[0].set_xlabel('insertion step (t)')
axes[0].set_ylabel('min distance to current bank')
axes[0].set_title('Nearest-other distance over progressive insertion')
axes[0].legend(loc='best', fontsize=9)

# Right: Probability of "YES" decision vs insertion step
axes[1].plot(x, p_yes_by_t, label='P(min < criterion)')
axes[1].set_xlabel('insertion step (t)')
axes[1].set_ylabel('probability')
axes[1].set_ylim(0, 1)
axes[1].set_title('Decision probability over progressive insertion')
axes[1].legend(loc='best', fontsize=9)

plt.tight_layout()
plt.show()

# 2) Histograms of min distances at selected steps across permutations
valid_times = [t for t in times_to_plot if 1 <= t <= N]
cols = len(valid_times)
fig2, axes2 = plt.subplots(1, cols, figsize=(4.2 * cols, 3.6), sharey=True)
if cols == 1:
    axes2 = [axes2]

# Choose common x-limit by high quantile over selected steps
pooled = np.concatenate([
    mins_by_t[t - 1, ~np.isnan(mins_by_t[t - 1])]
    for t in valid_times
    if np.any(~np.isnan(mins_by_t[t - 1]))
], axis=0) if len(valid_times) > 0 else np.array([])
x_max = float(np.percentile(pooled, 99.5)) if pooled.size > 0 else 1.0

for ax, t in zip(axes2, valid_times):
    data_t = mins_by_t[t - 1, ~np.isnan(mins_by_t[t - 1])]
    ax.hist(data_t, bins=40, range=(0, x_max), density=True, alpha=0.75)
    ax.axvline(crit, linestyle='--', linewidth=2, label=f'criterion = {crit:.3g}')
    ax.set_title(f'step t={t} | P(yes)={p_yes_by_t[t - 1]:.2f}')
    ax.set_xlabel('min distance')
    if t == valid_times[0]:
        ax.set_ylabel('density')
    ax.legend(fontsize=8, loc='upper right')

fig2.suptitle('Distribution of min distances across permutations (selected steps)', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# -*- coding: utf-8 -*-
# Progressive insertion with drift; plots BOTH cases at selected steps:
#   GREEN = "hit" probe (item IS already in bank)  -> distribution of min distances
#   RED   = "false-alarm" probe (item NOT in bank) -> distribution of min distances
#
# Assumptions:
# - DistanceMemoryModel.DistanceMemoryModel, zscore_projector, and test_sounds_list exist.
# - memory_model.noise_variance is the per-dimension *std* used per step (your class quirk).
# - We encode once to get clean reps X0; then simulate many permutations of progressive insertion.

import numpy as np
import torch
import matplotlib.pyplot as plt
import math

# ----------------------------- CONFIG ------------------------------------
nv = 0.165                                 # per-dim noise std for drift step
device = 'cuda' if torch.cuda.is_available() else 'cpu'
N_PERMS = 50                                # number of random insertion orders
SEED = 123                                   # set to None for stochastic runs
times_to_plot = [1, 5, 10, 16, 40, 80]        # 1-indexed steps to show in hist panels
CRIT_MULT = 1.5                                # criterion = CRIT_MULT * nv * sqrt(D)

# ----------------------------- SEEDING -----------------------------------
if SEED is not None:
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

# ------------------- ENCODE CLEAN REPRESENTATIONS ONCE -------------------
_tmp = DistanceMemoryModel.DistanceMemoryModel(
    encoding_model=zscore_projector,
    noise_variance=float(nv),
    criterion=0.0,
    device=device
)
_tmp._fill_memory_bank(test_sounds_list)                                  # use your pipeline
with torch.no_grad():
    clean_bank = [rep.detach().clone() for rep in _tmp.memory_bank]       # freeze clean
    X0 = torch.stack([rep.view(-1) for rep in clean_bank], dim=0).to(device)  # [N,D]
del _tmp
torch.cuda.empty_cache() if torch.cuda.is_available() else None

N, D = X0.shape                                                            # number of items, dims
crit = np.sqrt(D*ISI) * nv * 1 # distance threshold; tune via calibration if needed

# --------------------- STORAGE (separate hit vs FA) ----------------------
# For each step t (1..N), we keep *lists* of distances across perms:
fa_dists_by_t = [ [] for _ in range(N) ]                                   # FA: probe NOT in bank
hit_dists_by_t = [ [] for _ in range(N) ]                                  # Hit: probe IS in bank
fa_yes_rate = np.zeros(N, dtype=float)                                     # P(min<crit) for FA
hit_yes_rate = np.zeros(N, dtype=float)                                    # P(min<crit) for Hits
fa_counts = np.zeros(N, dtype=int)                                         # counts for FA per t
hit_counts = np.zeros(N, dtype=int)                                        # counts for Hits per t

# --------------------------- MAIN SIMULATION -----------------------------
for p in range(N_PERMS):
    # Fresh empty model for this permutation
    model = DistanceMemoryModel.DistanceMemoryModel(
        encoding_model=zscore_projector,
        noise_variance=float(nv),
        criterion=float(crit),
        device=device
    )
    model.memory_bank = []                                                 # start empty
    order = np.arange(N)                                                   # permutation of indices
    np.random.shuffle(order)

    with torch.no_grad():
        for t, idx_incoming in enumerate(order, start=1):                  # step t = 1..N
            # 1) Drift current bank by ONE step
            if len(model.memory_bank) > 0:
                model.apply_noise_to_memory()

            # 2) Build matrix view of CURRENT (drifted) bank; also clean counterparts
            if len(model.memory_bank) > 0:
                bank_mat = torch.stack([rep.view(-1) for rep in model.memory_bank], dim=0)  # [n,D]
                # clean reps for items already inserted are X0[order[:t-1]] in the SAME order
                X0_current = X0[order[:t-1]]                                                # [n,D]

                # ----- HIT CASE (probe IS in bank): clean probe vs current bank, all items -----
                # distance matrix clean-vs-drifted for all current items
                # cdist is convenient here and fast for small N (<=80)
                Dmat_hits = torch.cdist(X0_current, bank_mat, p=2)                           # [n,n]
                # For each item i, decision is min over the row (includes its drifted self)
                dmin_hits = Dmat_hits.min(dim=1).values                                      # [n]
                # Log per-item distances (adds many samples per step)
                hit_vals = dmin_hits.detach().cpu().numpy()
                hit_dists_by_t[t-1].extend(hit_vals.tolist())
                hit_yes_rate[t-1] += np.sum(hit_vals < crit)
                hit_counts[t-1] += hit_vals.size
            else:
                # At t=1, no hits are possible (empty bank)
                pass

            # 3) ----- FA CASE (probe NOT in bank): incoming CLEAN probe vs current bank -----
            probe_clean = X0[idx_incoming]                                                   # [D]
            if len(model.memory_bank) == 0:
                # bank empty -> undefined FA distance; skip logging
                pass
            else:
                dists_fa = torch.linalg.norm(bank_mat - probe_clean.view(1, -1), dim=1)      # [n]
                dmin_fa = float(dists_fa.min().item())
                fa_dists_by_t[t-1].append(dmin_fa)
                fa_yes_rate[t-1] += (dmin_fa < crit)
                fa_counts[t-1] += 1
            
            # 4) Insert the incoming CLEAN probe into the bank
            model.memory_bank.append(probe_clean.detach().clone())

# Normalize yes-rates by counts (avoid divide-by-zero)
fa_yes_rate = np.divide(fa_yes_rate, np.maximum(fa_counts, 1), dtype=float)
hit_yes_rate = np.divide(hit_yes_rate, np.maximum(hit_counts, 1), dtype=float)

# For time-course means/SEMs, compute over *samples* pooled across perms at each t
def _mean_sem_from_list(lst):
    arr = np.asarray(lst, dtype=float)
    if arr.size == 0:
        return np.nan, np.nan
    m = float(np.mean(arr))
    s = float(np.std(arr, ddof=1) / np.sqrt(len(arr))) if len(arr) > 1 else np.nan
    return m, s

hit_mean = np.zeros(N, dtype=float)
hit_sem  = np.zeros(N, dtype=float)
fa_mean  = np.zeros(N, dtype=float)
fa_sem   = np.zeros(N, dtype=float)
for t in range(N):
    m, s = _mean_sem_from_list(hit_dists_by_t[t]); hit_mean[t], hit_sem[t] = m, s
    m, s = _mean_sem_from_list(fa_dists_by_t[t]);  fa_mean[t],  fa_sem[t]  = m, s

# ------------------------------- PLOTS -----------------------------------
# A) Time courses: mean±SEM of min distance (green=hits, red=FAs) and P(yes)
x = np.arange(1, N+1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))

# Left panel: distance time-courses
axes[0].plot(x, hit_mean, label='hit: mean min dist', color='green')
axes[0].fill_between(x, hit_mean-hit_sem, hit_mean+hit_sem, alpha=0.25, color='green', label='hit ± SEM')
axes[0].plot(x, fa_mean,  label='FA: mean min dist',  color='red')
axes[0].fill_between(x, fa_mean-fa_sem, fa_mean+fa_sem, alpha=0.25, color='red', label='FA ± SEM')
axes[0].axhline(crit, linestyle='--', linewidth=2, color='k', label=f'criterion = {crit:.3g}')
axes[0].set_xlabel('insertion step (t)')
axes[0].set_ylabel('min distance to current bank')
axes[0].set_title('Nearest distance over progressive insertion')
axes[0].legend(loc='best', fontsize=9)

# Right panel: decision probabilities
axes[1].plot(x, hit_yes_rate, color='green', label='P(yes | hit probe)')
axes[1].plot(x, fa_yes_rate,  color='red',   label='P(yes | FA probe)')
axes[1].set_xlabel('insertion step (t)')
axes[1].set_ylabel('probability')
axes[1].set_ylim(0, 1)
axes[1].set_title('Decision probabilities over time')
axes[1].legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

# B) Histograms at selected steps (overlay green vs red)
valid_steps = [t for t in times_to_plot if 1 <= t <= N]
cols = len(valid_steps)
fig2, axes2 = plt.subplots(1, cols, figsize=(4.4*cols, 3.6), sharey=True)
if cols == 1:
    axes2 = [axes2]

# Choose common x-range by 99.5th percentile across both sets
pooled = []
for t in valid_steps:
    pooled.extend(hit_dists_by_t[t-1])
    pooled.extend(fa_dists_by_t[t-1])
pooled = np.asarray(pooled, dtype=float)
xmax = float(np.percentile(pooled, 99.5)) if pooled.size else 1.0

for ax, t in zip(axes2, valid_steps):
    hits_t = np.asarray(hit_dists_by_t[t-1], dtype=float)
    fas_t  = np.asarray(fa_dists_by_t[t-1], dtype=float)

    ax.hist(hits_t, bins=40, range=(0, xmax), density=True, alpha=0.55, color='green', label='hit (in bank)')
    ax.hist(fas_t,  bins=40, range=(0, xmax), density=True, alpha=0.55, color='red',   label='FA (not in bank)')
    ax.axvline(crit, linestyle='--', linewidth=2, color='k', label=f'criterion = {crit:.3g}')
    py_h = np.mean(hits_t < crit) if hits_t.size else 0.0
    py_f = np.mean(fas_t  < crit) if fas_t.size  else 0.0
    ax.set_title(f'step t={t} | Pyes(hit)={py_h:.2f} | Pyes(FA)={py_f:.2f}')
    ax.set_xlabel('min distance')
    if t == valid_steps[0]:
        ax.set_ylabel('density')
    ax.legend(fontsize=8, loc='upper right')

fig2.suptitle('Min-distance distributions by step (green=hit, red=FA)', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# -*- coding: utf-8 -*-
"""
False-Alarm curve with SERIAL insertion (order-driven ages), averaged over probes & orders.
- Probe is held out (never inserted).
- Bank of size T is built by serially inserting T items (random order), drifting existing items
  by ONE Brownian step before each insertion -> older traces drift more.
- We measure min L2 distance from the held-out probe to the drifted bank.
- Aggregate mean±SEM (and optionally P(yes)) vs T across permutations and probes.

Assumes your environment provides:
  - DistanceMemoryModel.DistanceMemoryModel
  - zscore_projector
  - test_sounds_list
"""

import numpy as np
import torch
import matplotlib.pyplot as plt

# ----------------------------- CONFIG ------------------------------------
NV = 0.6                                  # per-dim noise std for ONE drift step
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 123                                   # set to None for stochastic runs
N_PERMS = 50                                 # number of permutations (orders/probes)
MAX_PROBES_PER_PERM = 16                     # held-out probes sampled per permutation
T_VALUES = [1, 2, 4, 8, 12, 16, 24, 32, 48, 64]   # bank sizes to evaluate (must be < N)
REUSE_ORDER_PREFIXES = False                 # if True, reuse one order's prefixes across T
CRIT_MULT = 1.5                              # None -> no P(yes); else crit = c * NV * sqrt(D)

# ----------------------------- SEEDING -----------------------------------
if SEED is not None:
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

# ------------------------- ENCODE CLEAN REPS -----------------------------
# Encode once via your pipeline to obtain clean flattened reps X0 [N, D]
_tmp = DistanceMemoryModel.DistanceMemoryModel(
    encoding_model=zscore_projector,
    noise_variance=float(NV),
    criterion=0.0,
    device=DEVICE
)
_tmp._fill_memory_bank(test_sounds_list)  # your pipeline to compute embeddings
with torch.no_grad():
    clean_bank = [rep.detach().clone() for rep in _tmp.memory_bank]
    X0 = torch.stack([rep.view(-1) for rep in clean_bank], dim=0).to(DEVICE)  # [N, D]
del _tmp
if torch.cuda.is_available():
    torch.cuda.empty_cache()

N, D = X0.shape
crit = (CRIT_MULT * NV * np.sqrt(D)) if (CRIT_MULT is not None) else np.nan

# ---------------------------- STORAGE ------------------------------------
# Per-T collections of min distances (and optional yes/no decisions)
perT_dmins = {T: [] for T in T_VALUES}
perT_yes   = {T: [] for T in T_VALUES} if not np.isnan(crit) else None

# ------------------------------ SIM LOOP ---------------------------------
for _ in range(N_PERMS):
    # Optional CRN across T: one global order we reuse prefixes from
    global_order = np.random.permutation(N) if REUSE_ORDER_PREFIXES else None

    # Sample a set of held-out probes (unique per permutation)
    k = min(MAX_PROBES_PER_PERM, N)
    probe_indices = np.random.choice(N, size=k, replace=False)

    for probe_idx in probe_indices:
        # If reusing prefixes, drop the probe from the order once here
        if REUSE_ORDER_PREFIXES:
            order_wo_probe = global_order[global_order != probe_idx]

        for T in T_VALUES:
            if T <= 0 or T >= N:
                continue  # invalid bank size

            # Choose T bank indices excluding the probe
            if REUSE_ORDER_PREFIXES:
                bank_idx = order_wo_probe[:T]
            else:
                candidates = np.arange(N)
                candidates = candidates[candidates != probe_idx]
                bank_idx = np.random.choice(candidates, size=T, replace=False)

            # Build serial bank with drift-before-insert ages 1..T-1
            model = DistanceMemoryModel.DistanceMemoryModel(
                encoding_model=zscore_projector,
                noise_variance=float(NV),
                criterion=0.0,
                device=DEVICE
            )
            model.memory_bank = []

            with torch.no_grad():
                for j in range(T):
                    if len(model.memory_bank) > 0:
                        # One Brownian step for all existing items -> creates age gradient
                        model.apply_noise_to_memory()
                    # Insert a CLEAN copy (clone so model can mutate internally if needed)
                    model.memory_bank.append(X0[bank_idx[j]].detach().clone())

                # Compute min distance from held-out probe to the DRIFTED bank
                bank_mat = torch.stack([rep.view(-1) for rep in model.memory_bank], dim=0)  # [T,D]
                probe = X0[probe_idx].view(1, -1)                                           # [1,D]
                dists = torch.linalg.norm(bank_mat - probe, dim=1)                          # [T]
                dmin = float(dists.min().item())

            perT_dmins[T].append(dmin)
            if perT_yes is not None:
                perT_yes[T].append(float(dmin < crit))

# ---------------------------- AGGREGATION --------------------------------
T_vals = np.array(T_VALUES, dtype=int)

mean_min = np.empty(len(T_vals), dtype=float)
sem_min  = np.empty(len(T_vals), dtype=float)

if perT_yes is not None:
    pyes_mean = np.empty(len(T_vals), dtype=float)
    pyes_sem  = np.empty(len(T_vals), dtype=float)
else:
    pyes_mean = np.full(len(T_vals), np.nan, dtype=float)
    pyes_sem  = np.full(len(T_vals), np.nan, dtype=float)

for i, T in enumerate(T_vals):
    d = np.asarray(perT_dmins[T], dtype=float)
    # Mean and SEM of min distances; ddof=1 for unbiased std, guard for n<2
    mean_min[i] = float(np.mean(d)) if d.size else np.nan
    sem_min[i]  = float(np.std(d, ddof=1) / np.sqrt(d.size)) if d.size > 1 else np.nan

    if perT_yes is not None:
        y = np.asarray(perT_yes[T], dtype=float)
        pyes_mean[i] = float(np.mean(y)) if y.size else np.nan
        pyes_sem[i]  = float(np.std(y, ddof=1) / np.sqrt(y.size)) if y.size > 1 else np.nan

# ------------------------------- PLOTTING --------------------------------
if not np.isnan(crit):
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4.0))
else:
    fig, ax0 = plt.subplots(1, 1, figsize=(6.5, 4.0))
    ax1 = None

# Distance curve
ax0.plot(T_vals, mean_min, label='FA: mean min dist', color='r')
ax0.fill_between(T_vals, mean_min - sem_min, mean_min + sem_min, alpha=0.25, label='± SEM', color='r')
if not np.isnan(crit):
    ax0.axhline(crit, ls='--', lw=2, color='k', label=f'criterion = {crit:.3g}')
ax0.set_xlabel('bank size T (serial insertion; older traces drifted more)')
ax0.set_ylabel('min distance (probe vs bank)')
ax0.set_title('False-Alarm min distance vs T')
ax0.legend(loc='best')

# P(yes) curve (optional)
if ax1 is not None:
    ax1.plot(T_vals, pyes_mean, label='P(yes | FA probe)', color='r')
    ax1.fill_between(T_vals, pyes_mean - pyes_sem, pyes_mean + pyes_sem, alpha=0.25, label='± SEM', color='r')
    ax1.set_ylim(0, 1)
    ax1.set_xlabel('bank size T')
    ax1.set_ylabel('probability')
    ax1.set_title('Decision probability vs T')
    ax1.legend(loc='best')

plt.tight_layout()
plt.show()

In [ ]:
# -*- coding: utf-8 -*-
"""
Hits curve with SERIAL insertion (order-driven ages), averaged over probes & orders.

We compute THREE probe-age conditions to quickly bracket/illustrate behavior:
  A) 'probe-first'  : probe inserted FIRST  -> max age at end (~T-1 drifts)
  B) 'probe-middle' : probe inserted in the MIDDLE -> target non-zero middle age
  C) 'probe-last'   : probe inserted LAST   -> min age at end (0 drifts)

For each bank size T:
  - choose a probe index (the repeated item) and a permutation of the other T-1 items
  - build a serial-inserted bank; BEFORE each insertion, drift existing traces ONCE
  - measure min L2 distance between the CLEAN probe and the DRIFTED bank (includes its own drifted trace)
  - optionally compute P(yes) using crit = CRIT_MULT * NV * sqrt(D)

Assumes your environment provides:
  - DistanceMemoryModel.DistanceMemoryModel
  - zscore_projector
  - test_sounds_list
"""

import numpy as np
import torch
import matplotlib.pyplot as plt

# ----------------------------- CONFIG ------------------------------------
NV = 0.6                                 # per-dim noise std per ONE drift step (model quirk)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 123                                   # set to None for stochastic runs
N_PERMS = 50                                 # permutations (orders/probes batches)
MAX_PROBES_PER_PERM = 16                     # probe items sampled per permutation
T_VALUES = [1, 2, 4, 8, 12, 16, 24, 32, 48, 64]   # bank sizes (each must be <= N)
REUSE_ORDER_PREFIXES = False                 # if True, reuse one global order's prefixes across T
CRIT_MULT = 1.5                              # None -> skip P(yes); else crit = c * NV * sqrt(D)

# ----------------------------- SEEDING -----------------------------------
if SEED is not None:
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

# ------------------------- ENCODE CLEAN REPS -----------------------------
# Encode once to get clean flattened reps X0 [N, D]
_tmp = DistanceMemoryModel.DistanceMemoryModel(
    encoding_model=zscore_projector,
    noise_variance=float(NV),
    criterion=0.0,
    device=DEVICE
)
_tmp._fill_memory_bank(test_sounds_list)
with torch.no_grad():
    clean_bank = [rep.detach().clone() for rep in _tmp.memory_bank]                  # freeze clean copies
    X0 = torch.stack([rep.view(-1) for rep in clean_bank], dim=0).to(DEVICE)         # [N, D]
del _tmp
if torch.cuda.is_available():
    torch.cuda.empty_cache()

N, D = X0.shape
crit = (CRIT_MULT * NV * np.sqrt(D)) if (CRIT_MULT is not None) else np.nan          # distance criterion

# ---------------------------- STORAGE ------------------------------------
# Per-T collections for all three age cases
hit_first_dmins  = {T: [] for T in T_VALUES}   # probe FIRST
hit_middle_dmins = {T: [] for T in T_VALUES}   # probe MIDDLE (non-zero age)
hit_last_dmins   = {T: [] for T in T_VALUES}   # probe LAST

hit_first_yes  = {T: [] for T in T_VALUES} if not np.isnan(crit) else None
hit_middle_yes = {T: [] for T in T_VALUES} if not np.isnan(crit) else None
hit_last_yes   = {T: [] for T in T_VALUES} if not np.isnan(crit) else None

# ------------------------------ SIM LOOP ---------------------------------
for _ in range(N_PERMS):
    # Optional CRN across T: one global order reused (improves within-permutation pairing)
    global_order = np.random.permutation(N) if REUSE_ORDER_PREFIXES else None

    # Choose a batch of probes (unique per permutation)
    k = min(MAX_PROBES_PER_PERM, N)
    probe_indices = np.random.choice(N, size=k, replace=False)

    for probe_idx in probe_indices:
        # Build an "others" order excluding this probe (used when REUSE_ORDER_PREFIXES=True)
        if REUSE_ORDER_PREFIXES:
            others_order = global_order[global_order != probe_idx]  # order of non-probe items

        for T in T_VALUES:
            if T <= 0 or T > N:
                continue  # invalid bank size

            # ---- choose the same 'others_full' order for ALL three cases at this (probe, T) ----
            if REUSE_ORDER_PREFIXES:
                others_full = others_order[:max(T-1, 0)]
            else:
                cands = np.arange(N)
                cands = cands[cands != probe_idx]
                others_full = np.random.choice(cands, size=max(T-1, 0), replace=False)

            # ----------------- CASE A: probe FIRST (oldest by the end) -----------------
            model = DistanceMemoryModel.DistanceMemoryModel(
                encoding_model=zscore_projector,
                noise_variance=float(NV),
                criterion=0.0,
                device=DEVICE
            )
            model.memory_bank = []
            with torch.no_grad():
                # Insert probe first (no pre-drift since bank empty)
                model.memory_bank.append(X0[probe_idx].detach().clone())
                # Insert the remaining T-1 items, each preceded by ONE drift
                for idx in others_full:
                    model.apply_noise_to_memory()                          # ages probe each time
                    model.memory_bank.append(X0[idx].detach().clone())     # insert clean other

                # Special-case T==1: give the probe age=1 to avoid trivial zero distance
                if T == 1:
                    model.apply_noise_to_memory()

                # Measure min distance: CLEAN probe vs DRIFTED bank (includes its drifted self)
                bank_mat = torch.stack([rep.view(-1) for rep in model.memory_bank], dim=0)
                probe_clean = X0[probe_idx].view(1, -1)
                dmin_first = float(torch.linalg.norm(bank_mat - probe_clean, dim=1).min().item())
            hit_first_dmins[T].append(dmin_first)
            if hit_first_yes is not None:
                hit_first_yes[T].append(float(dmin_first < crit))

            # ----------------- CASE B: probe MIDDLE (target non-zero middle age) -----------------
            # Desired age for the probe at the end = number of insertions AFTER the probe
            # Choose middle_age = max(1, floor((T-1)/2)) to avoid age=0
            middle_age = max(1, (T - 1) // 2)                    # 1 .. T-1 (or 1 if T==1)
            n_after = min(middle_age, max(T - 1, 0))             # safety clamp
            n_before = max(T - 1 - n_after, 0)

            # Split others_full deterministically for pairing across cases
            others_before = others_full[:n_before]
            others_after  = others_full[n_before:n_before + n_after]

            model = DistanceMemoryModel.DistanceMemoryModel(
                encoding_model=zscore_projector,
                noise_variance=float(NV),
                criterion=0.0,
                device=DEVICE
            )
            model.memory_bank = []
            with torch.no_grad():
                # Insert 'before' items serially (drift before each insertion except the very first)
                for j, idx in enumerate(others_before):
                    if len(model.memory_bank) > 0:
                        model.apply_noise_to_memory()
                    model.memory_bank.append(X0[idx].detach().clone())

                # Insert the probe at this point (apply drift-before-insert rule if bank non-empty)
                if len(model.memory_bank) > 0:
                    model.apply_noise_to_memory()                 # ages 'before' items
                model.memory_bank.append(X0[probe_idx].detach().clone())  # probe age=0 at insertion

                # Insert 'after' items; each insertion drifts existing traces first -> ages probe
                for idx in others_after:
                    model.apply_noise_to_memory()                 # increments probe age by 1
                    model.memory_bank.append(X0[idx].detach().clone())

                # If T==1, give probe age=1 to avoid trivial zero distance
                if T == 1:
                    model.apply_noise_to_memory()

                # Measure min distance
                bank_mat = torch.stack([rep.view(-1) for rep in model.memory_bank], dim=0)
                probe_clean = X0[probe_idx].view(1, -1)
                dmin_middle = float(torch.linalg.norm(bank_mat - probe_clean, dim=1).min().item())
            hit_middle_dmins[T].append(dmin_middle)
            if hit_middle_yes is not None:
                hit_middle_yes[T].append(float(dmin_middle < crit))

            # ----------------- CASE C: probe LAST (youngest by the end; age=0) -----------------
            model = DistanceMemoryModel.DistanceMemoryModel(
                encoding_model=zscore_projector,
                noise_variance=float(NV),
                criterion=0.0,
                device=DEVICE
            )
            model.memory_bank = []
            with torch.no_grad():
                # Insert all T-1 others serially; drift before each insertion
                for j, idx in enumerate(others_full):
                    if len(model.memory_bank) > 0:
                        model.apply_noise_to_memory()
                    model.memory_bank.append(X0[idx].detach().clone())

                # Drift once BEFORE inserting probe to keep its final age at 0
                if len(model.memory_bank) > 0:
                    model.apply_noise_to_memory()
                model.memory_bank.append(X0[probe_idx].detach().clone())  # probe last, no post-drift

                # Measure min distance
                bank_mat = torch.stack([rep.view(-1) for rep in model.memory_bank], dim=0)
                probe_clean = X0[probe_idx].view(1, -1)
                dmin_last = float(torch.linalg.norm(bank_mat - probe_clean, dim=1).min().item())
            hit_last_dmins[T].append(dmin_last)
            if hit_last_yes is not None:
                hit_last_yes[T].append(float(dmin_last < crit))

# ---------------------------- AGGREGATION --------------------------------
T_vals = np.array(T_VALUES, dtype=int)

def _mean_sem(a):
    a = np.asarray(a, dtype=float)
    if a.size == 0:
        return np.nan, np.nan
    m = float(np.mean(a))
    s = float(np.std(a, ddof=1) / np.sqrt(a.size)) if a.size > 1 else np.nan
    return m, s

mean_first, sem_first   = np.empty(len(T_vals)), np.empty(len(T_vals))
mean_middle, sem_middle = np.empty(len(T_vals)), np.empty(len(T_vals))
mean_last, sem_last     = np.empty(len(T_vals)), np.empty(len(T_vals))

if not np.isnan(crit):
    p_first_m, p_first_s   = np.empty(len(T_vals)), np.empty(len(T_vals))
    p_middle_m, p_middle_s = np.empty(len(T_vals)), np.empty(len(T_vals))
    p_last_m, p_last_s     = np.empty(len(T_vals)), np.empty(len(T_vals))
else:
    p_first_m = p_first_s = p_middle_m = p_middle_s = p_last_m = p_last_s = np.full(len(T_vals), np.nan)

for i, T in enumerate(T_vals):
    m, s = _mean_sem(hit_first_dmins[T]);   mean_first[i],  sem_first[i]  = m, s
    m, s = _mean_sem(hit_middle_dmins[T]);  mean_middle[i], sem_middle[i] = m, s
    m, s = _mean_sem(hit_last_dmins[T]);    mean_last[i],   sem_last[i]   = m, s
    if not np.isnan(crit):
        m, s = _mean_sem(hit_first_yes[T]);   p_first_m[i],  p_first_s[i]  = m, s
        m, s = _mean_sem(hit_middle_yes[T]);  p_middle_m[i], p_middle_s[i] = m, s
        m, s = _mean_sem(hit_last_yes[T]);    p_last_m[i],   p_last_s[i]   = m, s

# ------------------------------- PLOTTING --------------------------------
if not np.isnan(crit):
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 4.6))
else:
    fig, ax0 = plt.subplots(1, 1, figsize=(7.5, 4.6))
    ax1 = None

# Distance curves
ax0.plot(T_vals, mean_first,  label='hit: probe-first (oldest)', color='green')
ax0.fill_between(T_vals, mean_first - sem_first,  mean_first + sem_first,  alpha=0.25, color='green')

ax0.plot(T_vals, mean_middle, label='hit: probe-middle (mid age)', color='mediumseagreen')
ax0.fill_between(T_vals, mean_middle - sem_middle, mean_middle + sem_middle, alpha=0.25, color='mediumseagreen')

ax0.plot(T_vals, mean_last,   label='hit: probe-last (youngest)', color='limegreen')
ax0.fill_between(T_vals, mean_last - sem_last,   mean_last + sem_last,   alpha=0.25, color='limegreen')

if not np.isnan(crit):
    ax0.axhline(crit, ls='--', lw=2, color='k', label=f'criterion = {crit:.3g}')

ax0.set_xlabel('bank size T (serial insertion)')
ax0.set_ylabel('min distance (clean probe vs drifted bank)')
ax0.set_title('Hit: min distance vs T (three probe-age cases)')
ax0.legend(loc='best')

# P(yes) curves (optional)
if ax1 is not None:
    ax1.plot(T_vals, p_first_m,  label='P(yes) | probe-first',  color='green')
    ax1.fill_between(T_vals, p_first_m - p_first_s,  p_first_m + p_first_s,  alpha=0.25, color='green')

    ax1.plot(T_vals, p_middle_m, label='P(yes) | probe-middle', color='mediumseagreen')
    ax1.fill_between(T_vals, p_middle_m - p_middle_s, p_middle_m + p_middle_s, alpha=0.25, color='mediumseagreen')

    ax1.plot(T_vals, p_last_m,   label='P(yes) | probe-last',   color='limegreen')
    ax1.fill_between(T_vals, p_last_m - p_last_s,   p_last_m + p_last_s,   alpha=0.25, color='limegreen')

    ax1.set_ylim(0, 1)
    ax1.set_xlabel('bank size T')
    ax1.set_ylabel('probability')
    ax1.set_title('Hit: decision probability vs T')
    ax1.legend(loc='best')

plt.tight_layout()
plt.show()